# Colab Evaluation Workflow

Run baseline and LoRA generation, then score TIFA and GenAI-Bench with API judges.

## 1. Environment Setup

In [ ]:
!mkdir -p /content/system_cache
!rm -rf /root/.cache
!ln -s /content/system_cache /root/.cache
!ls -l /root/.cache

lrwxrwxrwx 1 root root 21 Apr  9 17:10 /root/.cache -> /content/system_cache


In [ ]:
%cd /content/T2I-RL-Eval
!pip install -r requirements.txt

/content/T2I-RL-Eval
  Cloning https://github.com/deepseek-ai/Janus.git to /tmp/pip-install-vps1s2ch/janus_c603408c24ef4664a7a5e7939a411b3e
  Running command git clone --filter=blob:none --quiet https://github.com/deepseek-ai/Janus.git /tmp/pip-install-vps1s2ch/janus_c603408c24ef4664a7a5e7939a411b3e
  Resolved https://github.com/deepseek-ai/Janus.git to commit 1daa72fa409002d40931bd7b36a9280362469ead
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Configuration

In [ ]:
import os

API_KEY = ""
API_BASE_URL = "https://api.siliconflow.cn/v1"
JUDGE_MODEL = "Qwen/Qwen3.5-9B"

os.environ["OPENAI_API_KEY"] = API_KEY
os.environ["OPENAI_BASE_URL"] = API_BASE_URL
os.environ["JUDGE_MODEL"] = JUDGE_MODEL

BASE_MODEL = "deepseek-ai/Janus-Pro-1B"
LORA_DIR = "artifacts/lora/grpo_siliconflow_quick_final"
OUTPUT_DIR = "outputs/evaluation"

import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/T2I-RL-Eval")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## 3. Generate Before Images

In [ ]:
%cd /content/T2I-RL-Eval
!python scripts/generate_benchmark_images.py \
        --benchmark tifa --variant before \
        --manifest_path data/evaluation/tifa/samples.jsonl \
        --base_model $BASE_MODEL \
        --output_dir $OUTPUT_DIR \
        --resume \
        --dtype float16 \
        --prompt_batch_size 24

/content/T2I-RL-Eval
[generate] benchmark=tifa variant=before samples=4073 prompt_batch_size=24 dtype=float16 device=cuda
[generate] loading model=deepseek-ai/Janus-Pro-1B
Python version is above 3.10, patching the collections module.
2026-04-09 16:20:22.851539: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-09 16:20:22.870704: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775751622.893649   33132 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775751622.900464   33132 cuda_blas.cc:1407] Unable to register cuBLAS fac

In [ ]:
%cd /content/T2I-RL-Eval
!python scripts/generate_benchmark_images.py \
        --benchmark genai_bench --variant before \
        --manifest_path data/evaluation/genai_bench/samples.jsonl \
        --base_model $BASE_MODEL \
        --output_dir $OUTPUT_DIR \
        --resume \
        --dtype float16 \
        --prompt_batch_size 24

/content/T2I-RL-Eval
[generate] benchmark=genai_bench variant=before samples=1600 prompt_batch_size=24 dtype=float16 device=cuda
[generate] loading model=deepseek-ai/Janus-Pro-1B
Python version is above 3.10, patching the collections module.
2026-04-09 16:20:44.523136: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-09 16:20:44.541051: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775751644.563727   33222 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775751644.570438   33222 cuda_blas.cc:1407] Unable to register cuB

## 4. Generate After Images

In [ ]:
%cd /content/T2I-RL-Eval
!python scripts/generate_benchmark_images.py \
        --benchmark tifa --variant after \
        --manifest_path data/evaluation/tifa/samples.jsonl \
        --base_model $BASE_MODEL \
        --lora_path $LORA_DIR \
        --output_dir $OUTPUT_DIR \
        --resume \
        --dtype float16 \
        --prompt_batch_size 12

/content/T2I-RL-Eval
[generate] benchmark=tifa variant=after samples=4073 prompt_batch_size=12 dtype=float16 device=cuda
[generate] loading model=deepseek-ai/Janus-Pro-1B
Python version is above 3.10, patching the collections module.
2026-04-09 17:20:33.846602: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-09 17:20:33.866266: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775755233.889906   44438 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775755233.896683   44438 cuda_blas.cc:1407] Unable to register cuBLAS fact

In [ ]:
%cd /content/T2I-RL-Eval
!python scripts/generate_benchmark_images.py \
        --benchmark genai_bench --variant after \
        --manifest_path data/evaluation/genai_bench/samples.jsonl \
        --base_model $BASE_MODEL \
        --lora_path $LORA_DIR \
        --output_dir $OUTPUT_DIR \
        --resume \
        --dtype float16 \
        --prompt_batch_size 12

## 5. Run TIFA

In [ ]:
%cd /content/T2I-RL-Eval
!python scripts/run_tifa.py \
        --manifest_path data/evaluation/tifa/samples.jsonl \
        --images_root outputs/evaluation/images \
        --variant before \
        --output_path outputs/evaluation/results/tifa_before.jsonl \
        --resume \
        --max_workers 16 \
        --log_every 1
!python scripts/run_tifa.py \
        --manifest_path data/evaluation/tifa/samples.jsonl \
        --images_root outputs/evaluation/images \
        --variant after \
        --output_path outputs/evaluation/results/tifa_after.jsonl \
        --resume \
        --max_workers 16 \
        --log_every 1

## 6. Run GenAI-Bench

In [ ]:
%cd /content/T2I-RL-Eval
!python scripts/run_genai_bench.py \
        --manifest_path data/evaluation/genai_bench/samples.jsonl \
        --images_root outputs/evaluation/images \
        --variant before \
        --output_path outputs/evaluation/results/genai_before.jsonl \
        --resume \
        --max_workers 16 \
        --log_every 1
!python scripts/run_genai_bench.py \
        --manifest_path data/evaluation/genai_bench/samples.jsonl \
        --images_root outputs/evaluation/images \
        --variant after \
        --output_path outputs/evaluation/results/genai_after.jsonl \
        --resume \
        --max_workers 16 \
        --log_every 1

## 7. Summary

In [ ]:
%cd /content/T2I-RL-Eval
!python scripts/summarize_evaluation.py \
        --tifa_results_before outputs/evaluation/results/tifa_before.jsonl \
        --tifa_results_after outputs/evaluation/results/tifa_after.jsonl \
        --genai_results_before outputs/evaluation/results/genai_before.jsonl \
        --genai_results_after outputs/evaluation/results/genai_after.jsonl \
        --output_dir outputs/evaluation/reports